# Heart Disease UCI – Exploratory Data Analysis
**AIMLCZG523 MLOps Assignment 01 | BITS Pilani WILP**

This notebook covers:
1. Dataset loading & overview
2. Missing value analysis
3. Feature distributions (histograms)
4. Class balance
5. Correlation heatmap
6. Feature-target relationships

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# Load dataset (run data/download_data.py first, or use Colab upload)
df = pd.read_csv('../data/heart.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print(df.dtypes)
print('\n--- Descriptive Statistics ---')
df.describe().T

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0]

if missing_df.empty:
    print('No missing values found.')
else:
    print(missing_df)
    missing_df['missing_pct'].plot(kind='bar', color='salmon', title='Missing Values (%)')
    plt.tight_layout(); plt.show()

## 3. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['target'].value_counts()
axes[0].bar(['No Disease (0)', 'Disease (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Class Distribution – Count')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['No Disease', 'Disease'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Class Distribution – Proportion')

plt.suptitle('Target Class Balance', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Feature Distributions – Histograms

In [ ]:
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        axes[i].hist(df[df['target'] == label][col].dropna(),
                     bins=20, alpha=0.6, color=color,
                     label=f'Target={label}')
    axes[i].set_title(col)
    axes[i].legend()

axes[-1].axis('off')
plt.suptitle('Numeric Feature Distributions by Target', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Feature–Target Relationship Analysis

In [ ]:
# Box plots for numeric features split by target
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(18, 5))
for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='target', ax=axes[i], grid=False,
               patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='tomato', linewidth=2))
    axes[i].set_title(col)
    axes[i].set_xlabel('Target (0=No Disease, 1=Disease)')

plt.suptitle('Numeric Features vs Target', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Categorical features vs target
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = df.groupby([col, 'target']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=['steelblue', 'tomato'],
            edgecolor='white', legend=(i == 0))
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('Categorical Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Key EDA Findings

**Write your own observations here after running the notebook. For example:**
- Which features show the strongest correlation with the target?
- Is the dataset balanced or imbalanced?
- Which features have missing values and what imputation strategy makes sense?
- Any outliers in blood pressure or cholesterol?